## 04. Feature Engineering

### Objective

The objective of this notebook is to prepare the cleaned dataset for machine learning.

The main tasks include:

- Selecting relevant features.
- Removing unnecessary variables.
- Creating new meaningful features.
- Encoding categorical variables.
- Scaling numerical variables when appropriate.
- Exporting the final dataset for model training.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [2]:
data_path = Path(
    "../data/processed/telco_customer_churn_clean.csv"
)

df = pd.read_csv(data_path)

df.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        7043 non-null   str    
 1   gender             7043 non-null   str    
 2   senior_citizen     7043 non-null   str    
 3   partner            7043 non-null   str    
 4   dependents         7043 non-null   str    
 5   tenure             7043 non-null   int64  
 6   phone_service      7043 non-null   str    
 7   multiple_lines     7043 non-null   str    
 8   internet_service   7043 non-null   str    
 9   online_security    7043 non-null   str    
 10  online_backup      7043 non-null   str    
 11  device_protection  7043 non-null   str    
 12  tech_support       7043 non-null   str    
 13  streaming_tv       7043 non-null   str    
 14  streaming_movies   7043 non-null   str    
 15  contract           7043 non-null   str    
 16  paperless_billing  7043 non-null   

In [4]:
categorical_features = [
    "gender",
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract",
    "paperless_billing",
    "payment_method"
]

In [5]:
numerical_features = [
    "tenure",
    "monthly_charges",
    "total_charges"
]

target = "churn"

## Feature Selection

Feature selection determines which variables should be retained for machine learning.

The customer identifier will be removed because it does not describe customer behavior and does not provide useful predictive information.

In [6]:
df_model = df.drop(columns=["customer_id"]).copy()

df_model.head()

,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
selected_features = categorical_features + numerical_features + [target]

missing_features = [
    feature
    for feature in selected_features
    if feature not in df_model.columns
]

missing_features
assert len(missing_features) == 0, (
    f"Missing columns: {missing_features}"
)

In [8]:
classified_columns = set(
    categorical_features
    + numerical_features
    + [target]
)

unclassified_columns = [
    column
    for column in df_model.columns
    if column not in classified_columns
]

unclassified_columns

['senior_citizen']

In [9]:
categorical_features = [
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract",
    "paperless_billing",
    "payment_method"
]

In [10]:
classified_columns = set(
    categorical_features
    + numerical_features
    + [target]
)

unclassified_columns = [
    column
    for column in df_model.columns
    if column not in classified_columns
]

unclassified_columns

[]

In [11]:
df_model[categorical_features].dtypes

gender               str
senior_citizen       str
partner              str
dependents           str
phone_service        str
multiple_lines       str
internet_service     str
online_security      str
online_backup        str
device_protection    str
tech_support         str
streaming_tv         str
streaming_movies     str
contract             str
paperless_billing    str
payment_method       str
dtype: object

In [12]:
df_model[numerical_features].dtypes

tenure               int64
monthly_charges    float64
total_charges      float64
dtype: object

In [13]:
df_model[target].value_counts()

churn
No     5174
Yes    1869
Name: count, dtype: int64

## Feature Creation

New variables are created to summarize customer behavior and simplify patterns identified during exploratory analysis.

The new features focus on:

- Customer tenure stage.
- Contract commitment.
- Payment automation.
- Internet availability.
- Number of subscribed services.

In [14]:
df_model["tenure_group"] = pd.cut(
    df_model["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=[
        "0-12 months",
        "13-24 months",
        "25-48 months",
        "49-72 months"
    ]
)

In [15]:
df_model[
    ["tenure", "tenure_group"]
].head(10)

,tenure,tenure_group
0,1,0-12 months
1,34,25-48 months
2,2,0-12 months
3,45,25-48 months
4,2,0-12 months
5,8,0-12 months
6,22,13-24 months
7,10,0-12 months
8,28,25-48 months
9,62,49-72 months


In [16]:
df_model["tenure_group"].value_counts().sort_index()

tenure_group
0-12 months     2186
13-24 months    1024
25-48 months    1594
49-72 months    2239
Name: count, dtype: int64

In [17]:
df_model["long_term_contract"] = (
    df_model["contract"]
    .isin(["One year", "Two year"])
    .astype(int)
)

In [18]:
automatic_payment_methods = [
    "Bank transfer (automatic)",
    "Credit card (automatic)"
]

df_model["automatic_payment"] = (
    df_model["payment_method"]
    .isin(automatic_payment_methods)
    .astype(int)
)

In [19]:
df_model["has_internet"] = (
    df_model["internet_service"]
    .ne("No")
    .astype(int)
)

In [20]:
service_features = [
    "phone_service",
    "multiple_lines",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies"
]

In [21]:
df_model["number_of_services"] = (
    df_model[service_features]
    .eq("Yes")
    .sum(axis=1)
)

In [22]:
new_features = [
    "tenure_group",
    "long_term_contract",
    "automatic_payment",
    "has_internet",
    "number_of_services"
]

df_model[
    [
        "tenure",
        "contract",
        "payment_method",
        "internet_service",
        *new_features
    ]
].head(10)

,tenure,contract,payment_method,internet_service,tenure_group,long_term_contract,automatic_payment,has_internet,number_of_services
0,1,Month-to-month,Electronic check,DSL,0-12 months,0,0,1,1
1,34,One year,Mailed check,DSL,25-48 months,1,0,1,3
2,2,Month-to-month,Mailed check,DSL,0-12 months,0,0,1,3
3,45,One year,Bank transfer (automatic),DSL,25-48 months,1,1,1,3
4,2,Month-to-month,Electronic check,Fiber optic,0-12 months,0,0,1,1
5,8,Month-to-month,Electronic check,Fiber optic,0-12 months,0,0,1,5
6,22,Month-to-month,Credit card (automatic),Fiber optic,13-24 months,0,1,1,4
7,10,Month-to-month,Mailed check,DSL,0-12 months,0,0,1,1
8,28,Month-to-month,Electronic check,Fiber optic,25-48 months,0,0,1,6
9,62,One year,Bank transfer (automatic),DSL,49-72 months,1,1,1,3


In [23]:
for feature in new_features:
    print(f"\n{feature}")
    print(df_model[feature].value_counts(dropna=False))


tenure_group
tenure_group
49-72 months    2239
0-12 months     2186
25-48 months    1594
13-24 months    1024
Name: count, dtype: int64

long_term_contract
long_term_contract
0    3875
1    3168
Name: count, dtype: int64

automatic_payment
automatic_payment
0    3977
1    3066
Name: count, dtype: int64

has_internet
has_internet
1    5517
0    1526
Name: count, dtype: int64

number_of_services
number_of_services
1    1701
2    1188
3     965
4     922
5     908
6     676
7     395
8     208
0      80
Name: count, dtype: int64


In [24]:
assert df_model["tenure_group"].isna().sum() == 0

assert set(df_model["long_term_contract"].unique()).issubset({0, 1})
assert set(df_model["automatic_payment"].unique()).issubset({0, 1})
assert set(df_model["has_internet"].unique()).issubset({0, 1})

assert df_model["number_of_services"].between(
    0,
    len(service_features)
).all()

In [25]:
categorical_features.append("tenure_group")

In [26]:
engineered_numerical_features = [
    "long_term_contract",
    "automatic_payment",
    "has_internet",
    "number_of_services"
]

numerical_features.extend(engineered_numerical_features)

In [27]:
print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))

Categorical features: 17
Numerical features: 7


## Target Encoding

The target variable must be converted from text labels into numerical values before training machine learning models.

The following mapping will be used:

- No = 0
- Yes = 1

In [28]:
df_model["churn"] = df_model["churn"].map({
    "No": 0,
    "Yes": 1
})

In [29]:
df_model["churn"].value_counts()

churn
0    5174
1    1869
Name: count, dtype: int64

In [30]:
assert df_model["churn"].isna().sum() == 0
assert set(df_model["churn"].unique()) == {0, 1}

In [31]:
df_model[categorical_features].head()

,gender,senior_citizen,partner,dependents,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,tenure_group
0,Female,No,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,0-12 months
1,Male,No,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,25-48 months
2,Male,No,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,0-12 months
3,Male,No,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),25-48 months
4,Female,No,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,0-12 months


In [32]:
categorical_cardinality = pd.DataFrame({
    "feature": categorical_features,
    "unique_values": [
        df_model[feature].nunique()
        for feature in categorical_features
    ]
})

categorical_cardinality.sort_values(
    "unique_values",
    ascending=False
)

,feature,unique_values
16,tenure_group,4
15,payment_method,4
12,streaming_movies,3
8,online_backup,3
7,online_security,3
10,tech_support,3
9,device_protection,3
6,internet_service,3
5,multiple_lines,3
13,contract,3


## Feature Matrix and Target Vector

The dataset is separated into:

- `X`: predictor variables used by the models.
- `y`: target variable representing customer churn.

In [33]:
X = df_model.drop(columns=["churn"]).copy()
y = df_model["churn"].copy()

In [34]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 24)
y shape: (7043,)


In [35]:
assert X.shape[0] == y.shape[0]

## One-Hot Encoding

Categorical variables are transformed into binary indicator columns.

One-hot encoding avoids introducing artificial numerical order between categories.

In [36]:
X_encoded = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=False,
    dtype=int
)

In [37]:
drop="first"

In [38]:
print("Shape before encoding:", X.shape)
print("Shape after encoding:", X_encoded.shape)

Shape before encoding: (7043, 24)
Shape after encoding: (7043, 54)


In [39]:
X_encoded.head()

,tenure,monthly_charges,total_charges,long_term_contract,automatic_payment,has_internet,number_of_services,gender_Female,gender_Male,senior_citizen_No,...,paperless_billing_No,paperless_billing_Yes,payment_method_Bank transfer (automatic),payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,tenure_group_0-12 months,tenure_group_13-24 months,tenure_group_25-48 months,tenure_group_49-72 months
0,1,29.85,29.85,0,0,1,1,1,0,1,...,0,1,0,0,1,0,1,0,0,0
1,34,56.95,1889.50,1,0,1,3,0,1,1,...,1,0,0,0,0,1,0,0,1,0
2,2,53.85,108.15,0,0,1,3,0,1,1,...,0,1,0,0,0,1,1,0,0,0
3,45,42.30,1840.75,1,1,1,3,0,1,1,...,1,0,1,0,0,0,0,0,1,0
4,2,70.70,151.65,0,0,1,1,1,0,1,...,0,1,0,0,1,0,1,0,0,0


In [40]:
X_encoded.dtypes.value_counts()

int64      52
float64     2
Name: count, dtype: int64

In [41]:
non_numeric_columns = X_encoded.select_dtypes(
    exclude=["number"]
).columns.tolist()

non_numeric_columns

[]

In [42]:
assert len(non_numeric_columns) == 0

## Final Dataset Validation

Before exporting the modeling dataset, several validation checks are performed to confirm that:

- All features are numerical.
- No missing values remain.
- The target variable contains only binary values.
- The number of rows remains unchanged.
- No duplicated rows were introduced.

In [43]:
print("Encoded feature matrix shape:", X_encoded.shape)
print("Target vector shape:", y.shape)

print("\nMissing values in X:")
print(X_encoded.isna().sum().sum())

print("\nMissing values in y:")
print(y.isna().sum())

print("\nDuplicated rows in X:")
print(X_encoded.duplicated().sum())

print("\nTarget distribution:")
print(y.value_counts())

Encoded feature matrix shape: (7043, 54)
Target vector shape: (7043,)

Missing values in X:
0

Missing values in y:
0

Duplicated rows in X:
40

Target distribution:
churn
0    5174
1    1869
Name: count, dtype: int64


In [44]:
assert X_encoded.shape[0] == len(y)

assert X_encoded.isna().sum().sum() == 0
assert y.isna().sum() == 0

assert len(
    X_encoded.select_dtypes(exclude=["number"]).columns
) == 0

assert set(y.unique()) == {0, 1}

## Final Modeling Dataset

The encoded feature matrix and target variable are combined into a single dataset for export.

The target column is placed at the end to make the dataset easier to inspect and use in later notebooks.

In [45]:
modeling_df = pd.concat(
    [
        X_encoded.reset_index(drop=True),
        y.reset_index(drop=True)
    ],
    axis=1
)

In [46]:
modeling_df.head()

,tenure,monthly_charges,total_charges,long_term_contract,automatic_payment,has_internet,number_of_services,gender_Female,gender_Male,senior_citizen_No,...,paperless_billing_Yes,payment_method_Bank transfer (automatic),payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,tenure_group_0-12 months,tenure_group_13-24 months,tenure_group_25-48 months,tenure_group_49-72 months,churn
0,1,29.85,29.85,0,0,1,1,1,0,1,...,1,0,0,1,0,1,0,0,0,0
1,34,56.95,1889.50,1,0,1,3,0,1,1,...,0,0,0,0,1,0,0,1,0,0
2,2,53.85,108.15,0,0,1,3,0,1,1,...,1,0,0,0,1,1,0,0,0,1
3,45,42.30,1840.75,1,1,1,3,0,1,1,...,0,1,0,0,0,0,0,1,0,0
4,2,70.70,151.65,0,0,1,1,1,0,1,...,1,0,0,1,0,1,0,0,0,1


In [47]:
modeling_df.tail()

,tenure,monthly_charges,total_charges,long_term_contract,automatic_payment,has_internet,number_of_services,gender_Female,gender_Male,senior_citizen_No,...,paperless_billing_Yes,payment_method_Bank transfer (automatic),payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,tenure_group_0-12 months,tenure_group_13-24 months,tenure_group_25-48 months,tenure_group_49-72 months,churn
7038,24,84.80,1990.50,1,0,1,7,0,1,1,...,1,0,0,0,1,0,1,0,0,0
7039,72,103.20,7362.90,1,1,1,6,1,0,1,...,1,0,1,0,0,0,0,0,1,0
7040,11,29.60,346.45,0,0,1,1,1,0,1,...,1,0,0,1,0,1,0,0,0,0
7041,4,74.40,306.60,0,0,1,2,0,1,0,...,1,0,0,0,1,1,0,0,0,1
7042,66,105.65,6844.50,1,1,1,6,0,1,1,...,1,1,0,0,0,0,0,0,1,0


In [48]:
print("Final modeling dataset shape:", modeling_df.shape)

Final modeling dataset shape: (7043, 55)


In [49]:
print("Last column:", modeling_df.columns[-1])

Last column: churn


In [50]:
assert modeling_df.shape[0] == df.shape[0]
assert modeling_df.isna().sum().sum() == 0
assert modeling_df.columns[-1] == "churn"
assert set(modeling_df["churn"].unique()) == {0, 1}

In [51]:
modeling_df.duplicated().sum()

np.int64(22)

In [52]:
modeling_df.columns = (
    modeling_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

In [53]:
duplicated_column_names = (
    modeling_df.columns[
        modeling_df.columns.duplicated()
    ]
    .tolist()
)

duplicated_column_names

[]

In [54]:
assert len(duplicated_column_names) == 0

In [55]:
final_features = [
    column
    for column in modeling_df.columns
    if column != "churn"
]

print("Number of final features:", len(final_features))

Number of final features: 54


In [56]:
final_features[:20]

['tenure',
 'monthly_charges',
 'total_charges',
 'long_term_contract',
 'automatic_payment',
 'has_internet',
 'number_of_services',
 'gender_female',
 'gender_male',
 'senior_citizen_no',
 'senior_citizen_yes',
 'partner_no',
 'partner_yes',
 'dependents_no',
 'dependents_yes',
 'phone_service_no',
 'phone_service_yes',
 'multiple_lines_no',
 'multiple_lines_no_phone_service',
 'multiple_lines_yes']

## Export Modeling Dataset

The prepared dataset is exported to the processed data directory.

This file will be loaded in the machine learning notebook for train-test splitting, model training, and evaluation.

In [57]:
output_path = Path(
    "../data/processed/telco_customer_churn_modeling.csv"
)

In [58]:
modeling_df.to_csv(
    output_path,
    index=False
)

In [59]:
exported_df = pd.read_csv(output_path)

exported_df.head()

,tenure,monthly_charges,total_charges,long_term_contract,automatic_payment,has_internet,number_of_services,gender_female,gender_male,senior_citizen_no,...,paperless_billing_yes,payment_method_bank_transfer_automatic,payment_method_credit_card_automatic,payment_method_electronic_check,payment_method_mailed_check,tenure_group_0_12_months,tenure_group_13_24_months,tenure_group_25_48_months,tenure_group_49_72_months,churn
0,1,29.85,29.85,0,0,1,1,1,0,1,...,1,0,0,1,0,1,0,0,0,0
1,34,56.95,1889.50,1,0,1,3,0,1,1,...,0,0,0,0,1,0,0,1,0,0
2,2,53.85,108.15,0,0,1,3,0,1,1,...,1,0,0,0,1,1,0,0,0,1
3,45,42.30,1840.75,1,1,1,3,0,1,1,...,0,1,0,0,0,0,0,1,0,0
4,2,70.70,151.65,0,0,1,1,1,0,1,...,1,0,0,1,0,1,0,0,0,1


In [60]:
print("In-memory shape:", modeling_df.shape)
print("Exported shape:", exported_df.shape)

In-memory shape: (7043, 55)
Exported shape: (7043, 55)


In [61]:
assert exported_df.shape == modeling_df.shape
assert exported_df.columns.tolist() == modeling_df.columns.tolist()
assert exported_df.isna().sum().sum() == 0

## Feature Engineering Summary

The following transformations were completed:

- Removed `customer_id` because it does not provide predictive information.
- Classified variables into categorical and numerical groups.
- Created customer tenure groups.
- Created a long-term contract indicator.
- Created an automatic payment indicator.
- Created an internet service indicator.
- Counted the number of subscribed services.
- Encoded the churn target as a binary variable.
- Applied one-hot encoding to categorical features.
- Cleaned the generated column names.
- Validated the final modeling dataset.
- Exported the prepared dataset for machine learning.

Numerical scaling was intentionally postponed to the machine learning stage to prevent data leakage.

## Methodological Considerations

One-hot encoding was applied before exporting the modeling dataset because all categorical variables have relatively low cardinality.

Feature scaling was not performed in this notebook. Scaling must be fitted only on the training data after the train-test split. Applying scaling before splitting the dataset would allow information from the test set to influence the training process, resulting in data leakage.

Some engineered variables contain information derived from original variables. These variables are retained for initial model comparison and will later be evaluated using validation metrics and feature importance.